In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# DataFrame, який спільно використовують обидві клітинки
df = None

path_input = widgets.Text(
    value="",
    placeholder=r"C:\Users\User\Documents\data.csv",
    description="Шлях до файлу:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "110px"},
)

load_button = widgets.Button(
    description="Завантажити",
    button_style="success",
    icon="upload",
)

load_output = widgets.Output()

def on_load_clicked(_):
    global df
    with load_output:
        clear_output()
        path = path_input.value.strip().strip('"').strip("'")
        if not path:
            print("Помилка: введіть шлях до файлу.")
            return
        try:
            data = pd.read_csv(path)
        except FileNotFoundError:
            print(f'Помилка: файл "{path}" не знайдено. Перевірте шлях.')
            return
        except PermissionError:
            print(f'Помилка: немає доступу до файлу "{path}".')
            return
        except (pd.errors.ParserError, UnicodeDecodeError):
            print("Помилка: файл неможливо прочитати як CSV.")
            return
        except Exception as exc:
            print(f"Помилка при читанні файлу: {exc}")
            return

        if data.empty:
            print("Помилка: файл порожній, дані відсутні.")
            return
        if data.select_dtypes(include="number").empty:
            print("Помилка: у файлі відсутні числові стовпці для аналізу.")
            return

        df = data
        print(f"Дані успішно завантажено! Рядків: {len(df)}, стовпців: {len(df.columns)}")
        display(df.head())

load_button.on_click(on_load_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Крок 1. Завантаження даних</h3>"),
    widgets.HBox([path_input, load_button]),
    load_output,
]))

In [ ]:
import matplotlib.pyplot as plt

if df is None:
    display(widgets.HTML(
        "<b style='color:red'>Спочатку завантажте дані у першій клітинці "
        "(кнопка «Завантажити»), а потім повторно виконайте цю клітинку.</b>"
    ))
else:
    numeric_columns = list(df.select_dtypes(include="number").columns)

    # --- Віджети керування ---
    column_dropdown = widgets.Dropdown(
        options=numeric_columns,
        value=numeric_columns[0],
        description="Стовпець:",
    )
    width_slider = widgets.IntSlider(
        value=2, min=1, max=5, step=1,
        description="Товщина лінії:",
        continuous_update=False,
    )
    range_slider = widgets.IntRangeSlider(
        value=[0, len(df) - 1], min=0, max=len(df) - 1, step=1,
        description="Діапазон індексів:",
        continuous_update=False,
        layout=widgets.Layout(width="600px"),
        style={"description_width": "130px"},
    )

    # --- Віджети статистики ---
    count_label = widgets.Label()
    min_label = widgets.Label()
    max_label = widgets.Label()
    mean_label = widgets.Label()
    median_label = widgets.Label()

    plot_output = widgets.Output()

    def fmt(value):
        return f"{value:.2f}" if pd.notna(value) else "—"

    def update(*_):
        column = column_dropdown.value
        start, end = range_slider.value
        data = df[column].iloc[start:end + 1]

        count_label.value = f"Кількість значень: {int(data.count())}"
        min_label.value = f"Мінімальне значення: {fmt(data.min())}"
        max_label.value = f"Максимальне значення: {fmt(data.max())}"
        mean_label.value = f"Середнє значення: {fmt(data.mean())}"
        median_label.value = f"Медіана: {fmt(data.median())}"

        with plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.plot(range(start, end + 1), data.values,
                    linewidth=width_slider.value)
            ax.set_title(f"Графік стовпця «{column}»")
            ax.set_xlabel("Індекс")
            ax.set_ylabel(column)
            ax.grid(True)
            plt.show()

    column_dropdown.observe(update, names="value")
    width_slider.observe(update, names="value")
    range_slider.observe(update, names="value")

    stats_box = widgets.VBox(
        [widgets.HTML("<b>Статистичні характеристики:</b>"),
         count_label, min_label, max_label, mean_label, median_label],
        layout=widgets.Layout(
            border="1px solid gray", padding="10px",
            margin="0 0 0 20px", min_width="250px",
        ),
    )

    controls = widgets.VBox([column_dropdown, width_slider, range_slider])

    display(widgets.VBox([
        widgets.HTML("<h3>Крок 2. Інтерактивний графік</h3>"),
        widgets.HBox([controls, stats_box]),
        plot_output,
    ]))

    update()